# Evaluation and Error Analysis

Evaluate the trained models for Subject 01 and inspect where errors occur.
We focus on a few high-value checks: accuracy with imbalance-aware metrics, movement confusion, and boundary errors.


## What you will do
- Load Subject 01 test data and the trained models saved in the previous notebook.
- Compare overall performance using accuracy, balanced accuracy, and macro F1.
- Inspect rest vs movement errors and movement-to-movement confusions.
- Check whether errors cluster near label transitions.


In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
sns.set_style("whitegrid")


## Load Subject 01 data and trained models
The dataset contains standardized features and labels.
Models are loaded from the output directory created in notebook 04.


In [ ]:
DATA_PATH = Path("../data/processed/db1_subject01_win20_hop1.pkl")
MODEL_DIR = Path("../data/processed/models")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing dataset: {DATA_PATH}. Run notebook 03 to generate it."
    )
if not MODEL_DIR.exists():
    raise FileNotFoundError(
        f"Missing model directory: {MODEL_DIR}. Run notebook 04 to train models."
    )

with DATA_PATH.open("rb") as f:
    dataset = pickle.load(f)

X_test = dataset["X_test"]
y_test = dataset["y_test"]

subject_id = int(dataset["meta"]["subject_id"])
win_samples = int(dataset["meta"]["win_samples"])
hop_samples = int(dataset["meta"]["hop_samples"])


def _model_id(name: str) -> str:
    return name.lower().replace(" ", "_").replace("(", "").replace(")", "")


model_names = ["SVM (RBF)", "kNN", "MLP", "RF"]
models = {}
for name in model_names:
    model_path = (
        MODEL_DIR
        / f"db1_subject{subject_id:02d}_{_model_id(name)}_win{win_samples}_hop{hop_samples}.pkl"
    )
    if not model_path.exists():
        raise FileNotFoundError(
            f"Missing model file: {model_path}. Run notebook 04 to save it."
        )
    with model_path.open("rb") as f:
        models[name] = pickle.load(f)

X_test.shape

(216565, 30)

## Predict on the test set
We store predictions for each model so we can compare metrics consistently.


In [3]:
preds = {}
for name, model in models.items():
    preds[name] = model.predict(X_test)

sorted(preds.keys())

['MLP', 'SVM (RBF)', 'kNN']

## Overall metrics
Accuracy can be inflated by the rest class, so we also report balanced accuracy and macro F1.


In [ ]:
rows = []
for name, y_pred in preds.items():
    rows.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
            "macro_f1": f1_score(y_test, y_pred, average="macro"),
        }
    )

metrics_df = pd.DataFrame(rows).round(2).sort_values("macro_f1", ascending=False)
metrics_df

,model,accuracy,balanced_accuracy,macro_f1
2,MLP,0.82,0.62,0.64
1,kNN,0.79,0.55,0.57
0,SVM (RBF),0.78,0.49,0.56


Accuracy is the easiest number to read, but in NinaPro DB1 it is inflated by the large rest class (label 0). 
Balanced accuracy and macro F1 are more informative because they weight each class equally.

In our results, the MLP performs best across all three metrics, which suggests it generalizes better to the movement classes.
The gap between overall accuracy and balanced accuracy/macro F1 shows that recognizing individual movements is harder than the headline accuracy suggests.
This is why we rely on the confusion matrix and movement‑only metrics in the next steps.

## Per-class metrics (movement-focused)
We show a movement-only report (labels 1–52) to reduce the dominance of rest. The report includes the metrics:

- **precision**: Of the samples predicted as this class, how many are correct? (High precision = few false positives.)
- **recall**: Of the true samples of this class, how many did we find? (High recall = few false negatives.)
- **f1‑score**: Harmonic mean of precision and recall. (High F1 means both precision and recall are strong.)
- **support**: Number of true samples for that class in the test set. (Used for weighting averages; reflects class imbalance.)

In [5]:
best_model_name = metrics_df.iloc[0]["model"]
best_pred = preds[best_model_name]

labels_all = np.unique(y_test)
labels_movement = labels_all[labels_all > 0]

print(f"Best model by macro F1: {best_model_name}")
print("\nMovement-only classification report:\n")
print(
    classification_report(
        y_test,
        best_pred,
        labels=labels_movement,
        digits=3,
        zero_division=0,
    )
)


Best model by macro F1: MLP

Movement-only classification report:

              precision    recall  f1-score   support

           1      0.790     0.805     0.797      1920
           2      0.924     0.787     0.850      1485
           3      0.571     0.662     0.613      1820
           4      0.799     0.825     0.812      1460
           5      0.807     0.839     0.823      2008
           6      0.641     0.561     0.598      1493
           7      0.851     0.718     0.779      1289
           8      0.749     0.519     0.613      1512
           9      0.452     0.579     0.508      1371
          10      0.938     0.773     0.847      1395
          11      0.449     0.379     0.411      1337
          12      0.595     0.438     0.505      1715
          13      0.667     0.398     0.498      1825
          14      0.789     0.595     0.678      1766
          15      0.658     0.686     0.672      1514
          16      0.617     0.443     0.516      1581
          17  

## Movement difficulty by class and exercise
We sort movements by F1 score to see which are easiest or hardest to discriminate.

In [6]:
report = classification_report(
    y_test,
    best_pred,
    labels=labels_movement,
    digits=3,
    zero_division=0,
    output_dict=True,
)

rows = []
for label in labels_movement:
    entry = report[str(label)]
    rows.append(
        {
            "movement": int(label),
            "precision": entry["precision"],
            "recall": entry["recall"],
            "f1": entry["f1-score"],
            "support": int(entry["support"]),
        }
    )

movement_df = pd.DataFrame(rows)
movement_ranked = movement_df.sort_values("f1", ascending=False)
display(movement_ranked)

,movement,precision,recall,f1,support
1,2,0.924051,0.786532,0.849764,1485
9,10,0.938207,0.772760,0.847484,1395
51,52,0.864312,0.807292,0.834829,2304
4,5,0.806992,0.839143,0.822754,2008
3,4,0.798938,0.824658,0.811594,1460
18,19,0.827684,0.782377,0.804393,1498
0,1,0.789877,0.804688,0.797214,1920
6,7,0.851103,0.718386,0.779133,1289
40,41,0.752970,0.787675,0.769932,1931
44,45,0.768582,0.745018,0.756617,1957


## Rest vs movement errors
This 2x2 summary shows whether the model confuses rest with movement windows.


In [7]:
rest_true = (y_test == 0).astype(int)
rest_pred = (best_pred == 0).astype(int)

cm_rest = confusion_matrix(rest_true, rest_pred, labels=[0, 1])
cm_rest_df = pd.DataFrame(
    cm_rest,
    index=["true_movement", "true_rest"],
    columns=["pred_movement", "pred_rest"],
)
cm_rest_df


,pred_movement,pred_rest
true_movement,78233,12614
true_rest,4433,121285


## Top movement confusions
We list the most frequent movement-to-movement mistakes to identify ambiguous gestures.


In [8]:
cm_mov = confusion_matrix(y_test, best_pred, labels=labels_movement)
cm_mov = cm_mov.astype(np.int64)

confusions = []
for i, true_label in enumerate(labels_movement):
    row_total = cm_mov[i].sum()
    for j, pred_label in enumerate(labels_movement):
        if i == j:
            continue
        count = cm_mov[i, j]
        if count == 0:
            continue
        confusions.append(
            {
                "true": int(true_label),
                "pred": int(pred_label),
                "count": int(count),
                "share_of_true": count / row_total if row_total else 0.0,
            }
        )

conf_df = (
    pd.DataFrame(confusions)
    .sort_values(["count", "share_of_true"], ascending=False)
    .head(10)
)
conf_df


,true,pred,count,share_of_true
104,11,9,622,0.516611
93,9,11,426,0.327692
254,23,21,384,0.291572
165,16,17,271,0.221044
466,34,30,271,0.150975
115,13,9,247,0.172366
255,23,24,241,0.182992
819,51,50,241,0.111009
277,24,28,236,0.141064
406,31,39,228,0.129989


## Errors near label transitions
Windows around movement changes are often harder to classify.
We compare accuracy near transitions versus steady-state windows.


In [9]:
transition_idx = np.where(y_test[1:] != y_test[:-1])[0] + 1
radius = 5

mask_transition = np.zeros_like(y_test, dtype=bool)
for idx in transition_idx:
    start = max(0, idx - radius)
    end = min(len(y_test), idx + radius + 1)
    mask_transition[start:end] = True

mask_steady = ~mask_transition

acc_transition = accuracy_score(y_test[mask_transition], best_pred[mask_transition])
acc_steady = accuracy_score(y_test[mask_steady], best_pred[mask_steady])

pd.DataFrame(
    {
        "subset": ["near_transition", "steady_state"],
        "accuracy": [acc_transition, acc_steady],
    }
)


,subset,accuracy
0,near_transition,0.441759
1,steady_state,0.826332
